In [1]:
using Pkg
Pkg.activate(".")
using Distributed

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl`


In [2]:
num_workers = 6            # ← set to number of CPU cores you want to use
num_replicates = 30        # ← set as desired
addprocs(num_workers)

6-element Vector{Int64}:
 2
 3
 4
 5
 6
 7

In [3]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "median_RV.csv"
    data_column                  = "x1"
    scale_multiplier             = 1.0
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 2130
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "HAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "triweight"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "triweight"
    kernel_type_tvEWD            = "triweight"
    kernel_type_tvHAR            = "triweight"
    kernel_type_tvAR             = "triweight"

    alpha_level                  = 0.05
    verbose_output               = true
end

In [4]:
using CSV, DataFrames, BSON
# launch worker processes

# load essentials on each worker
@everywhere using Random, Statistics
@everywhere include("bootstrap_thresholds.jl")

      From worker 2:	WARNING: replacing module tvOLS_estimator.
      From worker 5:	WARNING: replacing module tvOLS_estimator.
      From worker 7:	WARNING: replacing module tvOLS_estimator.
      From worker 6:	WARNING: replacing module tvOLS_estimator.
      From worker 4:	WARNING: replacing module tvOLS_estimator.
      From worker 3:	WARNING: replacing module tvOLS_estimator.
      From worker 3:	WARNING: replacing module tvOLS_estimator.
      From worker 5:	WARNING: replacing module tvOLS_estimator.
      From worker 3:	WARNING: replacing module tvOLS_estimator.
      From worker 3:	WARNING: replacing module tvOLS_estimator.
      From worker 3:	WARNING: replacing module tvOLS_estimator.
      From worker 3:	WARNING: replacing module tvOLS_estimator.
      From worker 3:	WARNING: replacing module tvOLS_estimator.
      From worker 7:	WARNING: replacing module tvOLS_estimator.
      From worker 5:	WARNING: replacing module tvOLS_estimator.
      From worker 5:	WARNING: replacing 

In [5]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = scale_multiplier .* Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [6]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel_V2(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 5:	[ Info: Performing boostrap simulation number 4
      From worker 2:	[ Info: Performing boostrap simulation number 1
      From worker 4:	[ Info: Performing boostrap simulation number 3
      From worker 7:	[ Info: Performing boostrap simulation number 6
      From worker 3:	[ Info: Performing boostrap simulation number 2
      From worker 6:	[ Info: Performing boostrap simulation number 5
      From worker 7:	[ Info: Bootstrap 6 generated.
      From worker 7:	[ Info: Performing boostrap simulation number 7
      From worker 5:	[ Info: Bootstrap 4 generated.
      From worker 5:	[ Info: Performing boostrap simulation number 8
      From worker 6:	[ Info: Bootstrap 5 generated.
      From worker 6:	[ Info: Performing boostrap simulation number 9
      From worker 2:	[ Info: Bootstrap 1 generated.
      From worker 3:	[ Info: Bootstrap 2 generated.
      From worker 2:	[ Info: Performing boostrap simulation number 10
      From worker 3:	[ Info: Performing boostrap 

Task (done) @0x00000180cccb9b30

In [7]:
sed_vals

30-element Vector{Vector{Float64}}:
 [-3.431127665955525e-6, -2.6962952205428867e-6, -2.0764213213196023e-6, -1.5685040131187148e-6, -1.1688861594107972e-6, -8.733674836795248e-7, -6.772740649603628e-7, -5.757876507563669e-7, -5.642286275591267e-7, -6.379719907101193e-7  …  4.077707404163332e-6, 4.127858890915214e-6, 4.176887128352463e-6, 4.225257330256531e-6, 4.2735914136309905e-6, 4.322656737239315e-6, 4.373382477611737e-6, 4.426857276960142e-6, 4.484305425583716e-6, 4.547007693638541e-6]
 [2.6839254453012617e-5, 2.5293861522045686e-5, 2.369253680395397e-5, 2.2040867568684284e-5, 2.0344305382515344e-5, 1.8608144636913563e-5, 1.6837617234744518e-5, 1.5037796992682857e-5, 1.3213552811847528e-5, 1.1369524495383075e-5  …  -2.199796666428448e-6, -1.9855362974466394e-6, -1.7671366595907196e-6, -1.5446620520693328e-6, -1.3181915015523442e-6, -1.087815526854257e-6, -8.536172888732696e-7, -6.156842367356641e-7, -3.740862675145418e-7, -1.288779015448725e-7]
 [-1.2911655626186824e-6, -1.1848013

In [8]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

152

In [9]:
using StatsBase  # for quantile
function compute_global_threshold_V2(
        list_of_sed_vectors::Vector{Vector{Float64}},
        cutoff_start_index::Int,
        alpha_level::Float64 = 0.05
    )::Float64

    B = length(list_of_sed_vectors)
    out_of_sample_length = length(list_of_sed_vectors[1])
    time_cutoffs = Float64[]

    for t in cutoff_start_index:out_of_sample_length
        # collect the B values at time t
        raw_vals = [ list_of_sed_vectors[b][t] for b in 1:B ]
        # drop any NaNs
        clean_vals = filter(!isnan, raw_vals)

        # if everything was NaN you might skip or push a NaN
        if isempty(clean_vals)
            continue
        end

        push!(time_cutoffs, quantile(clean_vals, 1 - alpha_level))
    end

    return median(time_cutoffs)
end


compute_global_threshold_V2 (generic function with 2 methods)

In [10]:
thr = compute_global_threshold_V2(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 2.4282172814442012e-6

In [11]:
# will create sed_thresholds.bson in your working directory
BSON.@save "sed_thresholds.bson" sed_vals thr